# OU Quartic Double-Well — sim vs theory comparison

Driven by `theories/ou_quartic_colored.theory.py`.  The theory
is declared via the new `.equation(...)` DAE API, so the MF
solver in `pipeline._mean_field_dae` finds **all** roots of the
saddle equation and classifies their linear stability.  Two
operating regimes:

* **Single-well** (`mu > 0`): the only fixed point is `x* = 0`,
  stable.  Tree-level k=2 reduces to the standard linear-OU
  formula `⟨x(0)x(τ)⟩ = D/μ · exp(-μ|τ|)` plus a small `eps·x³`
  correction.
* **Double-well** (`mu < 0`): three roots — two stable wells at
  `x* = ±√(-μ/ε)` and one unstable saddle at `x* = 0`.
  `compute_cumulants(..., fixed_point_index=N)` picks which stable
  well to expand around (`0` = lower, `1` = upper).  The unstable
  origin appears in `th['mf_unstable_roots']` for inspection but
  is not a valid expansion point.

Set `FIXED_POINT_INDEX = None` in the config to read the theory's
METADATA default; override with an integer for a specific run.

## 1. Setup

In [ ]:
%display latex
%matplotlib inline

import os, sys, time, importlib, importlib.util
import numpy as np
import matplotlib.pyplot as plt

import os, sys
# --- depth-robust repo root: walk up until we find the 'pipeline' package ---
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'pipeline')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(os.path.join(_root, 'notebooks'))  # cwd=notebooks/ so relative data paths resolve as before

# Pipeline (theory side)
from pipeline import compute_cumulants

# Simulation side — linear-rate + hard-reset variant of the
# heterogeneous-pop simulator.  Same call signature as the other
# variants; only difference inside the Euler loop is
# ``λ = a·v`` (linear) and the hard reset ``v → 0`` per spike.
from models.ou_langevin_sim_numba import sim_ou_quartic_colored_numba
from models.cumulant_estimator import compute_kpoint_slice

## 2. Configuration

`fundamental` provides numerical values for every parameter in
`ou_quartic_colored.theory.py`.  Note the **bare**
parameter names (no `E` suffix) — there's only one population, so
no need to disambiguate: `tau` (membrane τ vector of size 2),
`mu` (linear restoring force), `eps` (cubic non-linearity), `D` (noise intensity).  All scalar parameters — no per-pop indexing.  MF saddle is `xstar = 0` for `mu > 0`.

Pick `k` / `max_ell` for the comparison:
- `k=2, max_ell=0` — tree-level cross-cumulant.  Linear φ ⇒ tree is
  the LNA prediction (exact for linear theories WITHOUT reset).  With
  reset, tree is still the LNA prediction in the no-reset limit;
  the 1-loop closes the reset-induced gap.
- `k=2, max_ell=1` — tree + 1-loop.  The 1-loop diagram comes from
  the reset's `(1,2)` vertex.  Sim should agree with `tree+loop`,
  not with `tree-only`.
- `k=1, max_ell=1` — tadpole shift to the mean rate.

In [ ]:
# Perturbative-regime defaults.  The dimensionless loop expansion
# parameter for this theory is roughly  g_eff = eps · D / mu² .
# At mu=1, eps=0.1, D=1:  g_eff = 0.1 (mildly perturbative).
# To push toward the bifurcation, lower mu — at mu → 0 the
# linear-OU variance D/mu blows up and the cubic eps·x³ term
# becomes the dominant restoring force.  At mu=0.1, eps=0.1, D=1
# the system is strongly nonlinear (g_eff = 10) and the
# perturbative series diverges — useful as a 'breakdown' test.
fundamental = {
    'mu':  1.0,
    'eps': 0.1,
    'D':   1.0,
    'tauc': 1.0,    # noise correlation time τc
}

k        = 2
# COST WARNING for double-well (mu < 0):
#   Phase J runtime explodes with |μ| at ell ≥ 1.  At μ=-1 ell=1
#   finishes in ~1s; at μ=-3 ell=1 takes >1 hour.  Reason: the m=3
#   poset modesum integrator handles polynomials in the vertex
#   coefficient 3·ε·x*² = 3·(-μ); that polynomial grows with |μ|
#   and the symbolic expansion blows up.  For double-well
#   exploration, prefer μ ≈ -1 with D ≈ 0.1 (barrier/D = 25,
#   trapped sim AND fast Phase J).
#   At xstar=0 (mu > 0) the action's xt·dx^2 vertex has coefficient
#   3·eps·xstar = 0, so the zero-prefactor short-circuit in
#   final_integral.py:3238 trivially skips most ell=2 subsets.
#   In that regime, Phase J at max_ell=2 finishes in seconds.
#
#   At xstar=±√(-mu/eps) the vertex is fully active, so every one
#   of the 48 ell=2 unique diagrams contributes a non-trivial m=5
#   polytope integration.  Phase J at max_ell=2 now takes ~hours
#   per τ point.  Drop to max_ell=1 (4 ell=1 diagrams, ~1s) for
#   double-well exploration unless you genuinely need the 2-loop
#   correction — the qualitative physics is captured at 1-loop.
max_ell  = 2
# This theory has a single field 'x' on a single population 'pop'
# of size 1, so the only sensible external label is ('x', 1).
if k == 1:
    external_fields = [('x', 1)]
else:
    external_fields = [('x', 1), ('x', 1)]

tau_max  = 30.0
tau_step = 1.0

# MF solver knobs (DAE path).  None ⇒ use METADATA defaults.
# When mu < 0 the saddle equation has 3 roots; FIXED_POINT_INDEX
# selects which STABLE root to expand around (0 = lower well,
# 1 = upper well).  The seed box widens the multi-start range
# so the Newton starts can reach the outer wells.
FIXED_POINT_INDEX = None
MF_DAE_SEED_BOX   = {'x': (-5.0, 5.0)}

PARALLEL  = True
N_WORKERS = None

# Phase J analytic flags (same toggles as the spike-reset notebook).
USE_GROUPED_PHASE_J       = False
USE_POLYGON_M2_INTEGRATOR = True
USE_POSET_INTEGRATOR      = True
USE_1D_INTEGRATOR         = True
USE_NUMBA_CHAIN_SIMPLEX   = True
USE_SIMPLIFY_FULL_IN_GT   = False

# Simulation knobs.  Euler-Maruyama is cheap per step.
N_RUNS   = 4
T_sim    = float(2_000_000)
dt_sim   = 0.05
dt_bin   = 0.5

print(f'k={k}, max_ell={max_ell}, external_fields={external_fields}')
print(f'tau_max={tau_max}, tau_step={tau_step}')
g_eff = fundamental['eps'] * fundamental['D'] / fundamental['mu']**2
print(f'dimensionless loop coupling g_eff = eps·D/mu² = {g_eff:.3f}')
print(f'  (perturbative when g_eff << 1; '
      f'breakdown / non-perturbative when g_eff > 1)')


## 3. Load the theory file

Single population E of size 2.  Fields are `n` (spike train) and
`v` (voltage), both with population annotation `'E'`.  Auto-response
names are `nt`, `vt`; auto-saddles are `nstar`, `vstar`.  The MF
equation for `vstar` is in closed form
`(Em + Σ w·g·nstar) / (1 + tau·nstar)`.

In [ ]:
THEORY_NAME = 'ou_quartic_colored'
THEORIES_DIR = os.path.abspath('../theories')

spec = importlib.util.spec_from_file_location(
    f'theories.{THEORY_NAME}',
    os.path.join(THEORIES_DIR, f'{THEORY_NAME}.theory.py'),
)
theory_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(theory_mod)

model = theory_mod.build()
print(f'Loaded theory: {model["name"]!r}')
print(f'Populations: {[(p["name"], p["size"]) for p in model["populations"]]}')
print(f'Fields (physical): '
      f'{[f["name"] for f in model["physical_fields"]]}')
print(f'Kernels: {[k["name"] for k in model["kernels"]]}')
print(f'Functions: '
      f'{[(fn["name"], fn.get("args")) for fn in model["functions"]]}')

### Optional: pre-compute the propagator + MF check

After loading the theory, you can call
`from pipeline import precompute; precompute(model)` once
to run the one-time structural pass (expand at
`taylor_order=2`, verify MF saddle cancels, build + cache the
propagator under `saved_theories/<theory>/`).  Subsequent
`compute_cumulants` calls — at any `taylor_order` — load the
propagator + MF saddle from cache instead of recomputing.

Same operation as the **Pre-compute** button in the theory-
builder UI (`notebooks/theory_builder.ipynb`).  Safe to skip;
the cache is also populated lazily on the first
`compute_cumulants` run.

## 3.5 Diagnostic: source + vertex types

Builds the FieldTheory expansion and prints every source vertex and
interaction vertex with its bigrade, leg structure, and **coefficient**
expression — both the raw form coming out of `decompose_sector` and a
`.simplify_full()`-cleaned form.  Hunt for two failure modes:

- **Coefficients that look big but should simplify** — e.g.
  `tau1*nstar1*phi1_1 - tau1*phi1_1*nstar1` should collapse to 0.  If
  the simplified column is much smaller than the raw column, an
  upstream substitution chain is leaving algebraic noise that the
  pipeline could simplify away.
- **Unsubstituted formal symbols** — `phi<k>_<i>`, `z_g_<i>_<j>`, or
  unresolved saddle vars in a source coefficient mean some
  substitution hook didn't fire for that index (same failure mode
  as the recent specializations `range(n_pop)` bug).

The diagnostic is cheap (just rebuilds `ft` and calls
`extract_*_types`) and runs in seconds, so you can iterate on it
without waiting for the full pipeline.

In [ ]:
from msrjd.core.field_theory import FieldTheory
from msrjd.core.vertices import (extract_source_types,
                                  extract_vertex_types,
                                  NoiseSourceType)
from sage.all import SR

# Build FT (uses the same Taylor budget the pipeline picks: max(k+2·ell, 2))
_taylor_order = max(k + 2 * max_ell, 2)
_ft_diag = FieldTheory(model, taylor_order=_taylor_order)
_ft_diag.expand()
print(f'FieldTheory.expand done (taylor_order={_taylor_order})')

_sources = extract_source_types(_ft_diag)
_vertices = extract_vertex_types(_ft_diag)
print(f'\n  {len(_sources)} source vertices, {len(_vertices)} interaction vertices')

# Group by bigrade for a quick summary.
from collections import Counter
src_bigrades = Counter(s.bigrade for s in _sources)
vt_bigrades  = Counter(v.bigrade for v in _vertices)
print(f'\n  Source bigrades:    {dict(sorted(src_bigrades.items()))}')
print(f'  Interaction bigrades: {dict(sorted(vt_bigrades.items()))}')

def _coeff_diag(label, idx, obj):
    """Print the raw vs simplified coefficient for one source/vertex."""
    raw = SR(obj.coefficient)
    raw_str = str(raw)
    try:
        simp = raw.simplify_full()
    except Exception:
        simp = raw
    simp_str = str(simp)
    raw_len = len(raw_str)
    simp_len = len(simp_str)
    legs_str = (f'resp={obj.response_legs}'
                + (f', phys={obj.physical_legs}'
                   if hasattr(obj, 'physical_legs') else ''))
    print(f'  {label}[{idx}]  bigrade={obj.bigrade}  {legs_str}')
    print(f'    raw  ({raw_len:4d} chars): {raw_str}')
    if simp_str != raw_str:
        delta = raw_len - simp_len
        sign  = '−' if delta > 0 else '+'
        print(f'    simp ({simp_len:4d} chars, {sign}{abs(delta):3d}): {simp_str}')
    if isinstance(obj, NoiseSourceType):
        print(f'    cumulant_specs: {len(obj.cumulant_specs)} '
              f'({", ".join(s["noise"] for s in obj.cumulant_specs)})')
    # Flag residual formal-rename symbols (these should be substituted
    # by the time we get here — if they're still here, an action-side
    # substitution hook missed an index).
    residual = []
    for v in raw.variables():
        sname = str(v)
        if sname.startswith(('phi', 'z_g_', '_v_mf_', '_phi_arg_')):
            residual.append(sname)
    if residual:
        print(f'    ⚠ residual formal symbols: {sorted(set(residual))}')

print('\n=== Source vertices ===')
for idx, src in enumerate(_sources):
    _coeff_diag('src', idx, src)

print('\n=== Interaction vertex types ===')
for idx, vt in enumerate(_vertices):
    _coeff_diag('vt ', idx, vt)

## 3.6 Performance diagnostics (optional)

These cells help isolate slow paths.  Skip on production runs.

**Cell A** A/Bs parallel vs serial on a tiny `k=1, max_ell=1`
sub-config (~30s) so you can confirm `multiprocessing.Pool` is
actually firing and that bit-identity holds.

**Cell B** runs `cProfile` over a 1-point τ-grid with your
current `k`, `max_ell`.  Prints the top 40 frames by cumulative
time and writes a `.prof` file you can open with `snakeviz`.

**Cell C** runs the analytic-path runtime counters: distinguishes
intent (the `subset_evaluators` labels that get set at setup time)
from runtime (whether the analytic path actually completed or
silently fell back to scipy.nquad).  Lightweight (~half a normal
run).  Use this to confirm Stage 3b-extended is firing on m≥3
subsets after a kernel restart.

If `max_ell=2` and a full grid is too slow to wait for, run Cell B
with `_PROF_MAX_ELL` set to the current value — the 1-point τ-grid
keeps it bounded even when the per-τ cost is high.


In [ ]:
# === Cell A: parallelization sanity check ===
# Times parallel vs serial on a quick (k=1, max_ell=1) sub-config
# so we can confirm multiprocessing.Pool is actually firing on this
# machine.  Should take ~30 seconds total.

# Disabled by default — flip ``False`` → ``True`` to run.
if False:
    import os, time
    import numpy as np
    from msrjd.integration.time_domain import final_integral as _fi

    # Wire analytic-integrator flags first.
    _fi.USE_POLYGON_M2_INTEGRATOR = USE_POLYGON_M2_INTEGRATOR
    _fi.USE_POSET_INTEGRATOR      = USE_POSET_INTEGRATOR
    _fi.USE_1D_INTEGRATOR         = USE_1D_INTEGRATOR
    _fi.USE_NUMBA_CHAIN_SIMPLEX   = USE_NUMBA_CHAIN_SIMPLEX
    from msrjd.integration.time_domain import propagator_td as _pt
    _pt.USE_SIMPLIFY_FULL_IN_GT   = USE_SIMPLIFY_FULL_IN_GT

    print(f'os.cpu_count()                = {os.cpu_count()}')
    print(f'N_WORKERS (None ⇒ auto)       = {N_WORKERS}')
    print()

    # Small config: enough τ points to fill the worker pool.
    _diag_cfg = dict(
        model               = model,
        k                   = 1,
        max_ell             = 1,
        fundamental         = fundamental,
        external_fields     = [('x', 1)],
        tau_max             = 5.0,
        tau_step            = 0.5,           # 11 τ points
        n_workers           = N_WORKERS,
        verbose             = False,
        use_grouped_phase_j = False,
    )

    t0 = time.perf_counter()
    _th_ser = compute_cumulants(**_diag_cfg, parallel=False)
    _t_ser = time.perf_counter() - t0

    t0 = time.perf_counter()
    _th_par = compute_cumulants(**_diag_cfg, parallel=True)
    _t_par = time.perf_counter() - t0

    _n_tau = len(_th_ser['tau_grid'])
    _n_cpu = os.cpu_count() or 1
    _ideal = min(_n_cpu, _n_tau)
    _speedup = _t_ser / max(_t_par, 1e-9)
    _diff = np.max(np.abs(_th_par['C_tau'] - _th_ser['C_tau']))

    print(f'serial    : {_t_ser:6.2f}s   ({_n_tau} τ points)')
    print(f'parallel  : {_t_par:6.2f}s')
    print(f'speedup   : {_speedup:.2f}x   '
          f'(ideal ≈ min(cpu_count={_n_cpu}, n_tau={_n_tau}) = {_ideal})')
    print(f'|C_par − C_ser|_max = {_diff:.2e}  '
          f'{"OK" if _diff < 1e-12 else "⚠ bit-identity broken"}')
    print()
    if _speedup < 1.5:
        print('⚠  parallelism does not seem to be firing.  Common causes:')
        print('   • total_tasks < max(4, 2*n_workers): too small to amortise')
        print('     pool setup — see pipeline.total_C_batch serial fast path.')
        print('   • fork start method rejected (rare on macOS/Linux).')
        print('   • PARALLEL=False at the top level (check this notebook).')
    elif _speedup < _ideal * 0.5:
        print(f'⚠  speedup ({_speedup:.1f}x) is well below ideal '
              f'({_ideal}x) — pool setup + GIL contention from outside')
        print('   workers may be dominating.  Worth a profile.')
    else:
        print(f'✓ parallelism healthy: {_speedup:.1f}x of {_ideal}x ideal.')


In [ ]:
# === Cell B: cProfile k, max_ell on a 1-point τ grid ===
# Profiles the SERIAL path (cProfile can't see worker processes).
# Uses the current k / max_ell so we see the actual bottleneck
# you care about.  The 1-point grid keeps wall time bounded even
# at 2-loop.

# Disabled by default — flip ``False`` → ``True`` to run.
if False:
    import cProfile, pstats, io, time

    _PROF_K       = k
    _PROF_MAX_ELL = max_ell
    print(f'Profiling k={_PROF_K}, max_ell={_PROF_MAX_ELL}, 1 τ point '
          f'(serial; cProfile does not see workers)...')

    _pr = cProfile.Profile()
    _t0 = time.perf_counter()
    _pr.enable()
    _th_prof = compute_cumulants(
        model               = model,
        k                   = _PROF_K,
        max_ell             = _PROF_MAX_ELL,
        fundamental         = fundamental,
        external_fields     = external_fields,
        tau_max             = 0.0,           # → grid is [0.0]
        tau_step            = 1.0,           # 1 τ point total
        parallel            = False,
        verbose             = False,
        use_grouped_phase_j = USE_GROUPED_PHASE_J,
    )
    _pr.disable()
    print(f'Profile wall time (serial, 1 τ point): '
          f'{time.perf_counter() - _t0:.1f}s')
    print()

    _buf = io.StringIO()
    pstats.Stats(_pr, stream=_buf).sort_stats('cumulative').print_stats(40)
    print(_buf.getvalue())

    _prof_path = 'phase_j_profile.prof'
    _pr.dump_stats(_prof_path)
    print(f'Saved {_prof_path} — open with `snakeviz {_prof_path}` for '
          f'a flame view.')
    print()
    print('What to look for in the top-40 cumulative list:')
    print("  • '_integrate_nd_polytope' / scipy quadpack dominating")
    print('    ⇒ degenerate-β closed form is the right next step.')
    print("  • '_build_fast_subset_evaluator_from_modes' dominating")
    print('    ⇒ cache compiled callables per subset signature.')
    print("  • 'SR.subs' / 'fast_callable' compile dominating")
    print('    ⇒ Stage 4 subset-signature plan cache pays off.')
    print("  • '_enumerate_pole_tuples' or polygon loop dominating")
    print('    ⇒ vectorise the multi-index loop with numpy.')


In [ ]:
# === Cell C: analytic-path runtime counters ===
# The `subset_evaluators` breakdown in the main run cell reports
# INTENT (the evaluator label set at subset setup time), not
# whether the analytic path actually completed at runtime.  When an
# analytic path returns None (degenerate β past the polynomial
# extension, mixed constraint, maximality failure, etc.), the
# closure silently falls back to scipy.nquad — same label, very
# different wall time.
#
# This cell zeros the runtime counters, runs a single-τ pass with
# your current k / max_ell, and prints what actually fired.  Mirrors
# the normal pipeline call but with parallel=False and a 1-point
# τ-grid (tau_max=0.0) so it's bounded.

# Disabled by default — flip ``False`` → ``True`` to run.
if False:
    from msrjd.integration.time_domain import final_integral as _fi
    import time

    _fi.USE_POLYGON_M2_INTEGRATOR = USE_POLYGON_M2_INTEGRATOR
    _fi.USE_POSET_INTEGRATOR      = USE_POSET_INTEGRATOR
    _fi.USE_1D_INTEGRATOR         = USE_1D_INTEGRATOR
    _fi.USE_NUMBA_CHAIN_SIMPLEX   = USE_NUMBA_CHAIN_SIMPLEX
    from msrjd.integration.time_domain import propagator_td as _pt
    _pt.USE_SIMPLIFY_FULL_IN_GT   = USE_SIMPLIFY_FULL_IN_GT
    _fi._reset_runtime_counters()

    # k=1 needs exactly 1 external field; k≥2 needs k of them.
    _DIAG_EXT = ([('x', 1)] if k == 1
                 else [('n', i + 1) for i in range(k)])

    print(f'Running diagnostic: k={k}, max_ell={max_ell}, 1 τ point '
          f'(serial; counters track every analytic-path decision)...')
    _t0 = time.perf_counter()
    _th_diag = compute_cumulants(
        model               = model,
        k                   = k,
        max_ell             = max_ell,
        fundamental         = fundamental,
        external_fields     = _DIAG_EXT,
        tau_max             = 0.0,           # → grid is [0.0]
        tau_step            = 1.0,           # 1 τ point total
        parallel            = False,
        verbose             = False,
        use_grouped_phase_j = USE_GROUPED_PHASE_J,
    )
    _wall = time.perf_counter() - _t0
    print(f'Wall time (serial, single-τ): {_wall:.1f}s')
    print()

    # Pretty-print counters in grouped form for readability.
    _c = dict(_fi._RUNTIME_COUNTERS)
    print('=== Runtime path counters ===')
    print(f'  m=1 interval path:  attempted={_c["interval_attempted"]:>6d}'
          f'  none={_c["interval_returned_none"]:>6d}')
    print(f'  m=2 polygon  path:  attempted={_c["polygon_attempted"]:>6d}'
          f'  none={_c["polygon_returned_none"]:>6d}')
    print(f'  m≥3 poset    path:  attempted={_c["poset_attempted"]:>6d}'
          f'  none={_c["poset_returned_none_total"]:>6d}')
    print('    breakdown of m≥3 None returns:')
    print(f'      extract returned None:   '
          f'{_c["poset_extract_returned_none"]:>6d}')
    print(f'      consistent_lower failed: '
          f'{_c["poset_consistent_lower_failed"]:>6d}')
    print(f'      maximality check failed: '
          f'{_c["poset_maximality_failed"]:>6d}')
    print(f'      chain fast → polynomial: '
          f'{_c["chain_simplex_fast_returned_none"]:>6d}')
    print(f'      chain polynomial None:   '
          f'{_c["chain_simplex_polynomial_returned_none"]:>6d}')
    print()
    print(f'  scipy.nquad fallback:  m=1={_c["scipy_nquad_called_m1"]:>6d}'
          f'   m=2={_c["scipy_nquad_called_m2"]:>6d}'
          f'   m≥3={_c["scipy_nquad_called_mge3"]:>6d}')
    print()

    # Cheap diagnosis cribsheet.
    _n_m3 = _c['poset_attempted']
    _n_fb = _c['scipy_nquad_called_mge3']
    if _n_m3 == 0:
        print('No m≥3 subsets exercised at this (k, max_ell).')
    elif _n_fb == 0:
        print(f'✓ All {_n_m3} m≥3 subsets handled analytically — '
              f'Stage 3b/3b-extended is doing its job.')
        print(f'  If the run is still slow, the bottleneck is the analytic '
              f'path itself (numba/numpy vectorization next).')
    elif _n_fb >= _n_m3 * 0.9:
        print(f'⚠ Nearly all m≥3 subsets ({_n_fb}/{_n_m3}) silently fell '
              f'back to scipy.nquad.  See breakdown above for the dominant')
        print('  None-return cause.')
    else:
        print(f'Partial coverage: {_n_m3 - _n_fb}/{_n_m3} analytic, '
              f'{_n_fb}/{_n_m3} scipy fallback.  Mixed; check breakdown.')


## 4. Theory side — one pipeline call

Runs the full MSR-JD chain.  At `max_ell=0` the tree-level
$C^{(2)}(\tau)$ matches the LNA-without-reset prediction; at
`max_ell=1` the 1-loop contribution from the reset's `(1, 2)` vertex
is added.  Symbolic propagator is cached under
`saved_theories/ou_quartic_colored_taylor4/`.

In [ ]:
# Wire the module-level analytic-integrator flags from the config above.
# (compute_cumulants doesn't accept these as kwargs — they're global
# behaviour switches on the final-integral module.)
from msrjd.integration.time_domain import final_integral as _fi
_fi.USE_POLYGON_M2_INTEGRATOR = USE_POLYGON_M2_INTEGRATOR
_fi.USE_POSET_INTEGRATOR      = USE_POSET_INTEGRATOR
_fi.USE_1D_INTEGRATOR         = USE_1D_INTEGRATOR
_fi.USE_NUMBA_CHAIN_SIMPLEX   = USE_NUMBA_CHAIN_SIMPLEX
from msrjd.integration.time_domain import propagator_td as _pt
_pt.USE_SIMPLIFY_FULL_IN_GT   = USE_SIMPLIFY_FULL_IN_GT

t0 = time.perf_counter()
th = compute_cumulants(
    model               = model,
    k                   = k,
    max_ell             = max_ell,
    fundamental         = fundamental,
    external_fields     = external_fields,
    tau_max             = tau_max,
    tau_step            = tau_step,
    parallel            = PARALLEL,
    n_workers           = N_WORKERS,
    use_grouped_phase_j = USE_GROUPED_PHASE_J,
    fixed_point_index   = (FIXED_POINT_INDEX if FIXED_POINT_INDEX is not None else 0),
    mf_dae_seed_box     = MF_DAE_SEED_BOX,
    verbose             = True,
)
print(f'\nTheory side took {time.perf_counter() - t0:.1f}s')

tau_grid_th    = th['tau_grid']
C_theory_total = th['C_tau'].real
C_by_ell       = th['C_tau_by_ell']
C_theory_tree  = (C_by_ell[0].real if 0 in C_by_ell
                  else np.zeros_like(C_theory_total))
C_theory_loop  = C_theory_total - C_theory_tree

mf_values = th['mf_values']
print('\nMean-field saddles:')
for name, vals in mf_values.items():
    print(f'  {name!r:8} = {vals}')

# Pull the saddle the DAE solver chose to expand around.  For mu>0
# this is just xstar = 0 (the single stable root); for mu<0 it's
# one of the two stable wells, picked by FIXED_POINT_INDEX above.
xstar = np.array(mf_values.get('xstar', [0.0]))
print(f'\nxstar saddle (selected) = {xstar.tolist()}')

# Multi-root report (new DAE machinery).  Single-well runs show
# 1 stable / 0 unstable; double-well runs show 2 stable + 1
# unstable (the origin).
all_roots      = th.get('mf_all_roots', [])
stable_roots   = th.get('mf_stable_roots', [])
unstable_roots = th.get('mf_unstable_roots', [])
if all_roots:
    print(f'\nMF DAE solver found {len(all_roots)} fixed point(s): '
          f'{len(stable_roots)} stable, {len(unstable_roots)} unstable.')
    for i, entry in enumerate(all_roots):
        r    = entry['values']
        eigs = ', '.join(f'{e.real:+.4f}' for e in entry['eigenvalues_finite'])
        cls  = 'STABLE  ' if entry['stable'] else 'UNSTABLE'
        tag  = ' ← selected' if entry['stable'] and r['xstar'] == xstar.tolist() else ''
        print(f'    [{i}] x*={r["xstar"][0]:+.4f}  {cls}  σ=[{eigs}]{tag}')
    if len(stable_roots) > 1:
        print(f'\n  (override FIXED_POINT_INDEX above to expand around '
              f'a different stable branch; valid range: '
              f'0…{len(stable_roots)-1})')

print(f'\nTotal diagrams: {len(th["diagrams"])}')
n_per_ell = {ell: sum(1 for r in th['diagrams'] if r['ell'] == ell)
             for ell in sorted({r['ell'] for r in th['diagrams']})}
for ell, n_d in n_per_ell.items():
    print(f'    ell={ell}: {n_d} diagrams')

# ─── Phase J evaluator breakdown (Stage 3a + 3b diagnostic) ──────────
# Which numerical evaluator path did each per-diagram subset use?
#   'polygon_modesum' = Stage 3a-full analytic m=2 integrator fired.
#   'poset_modesum'   = Stage 3b causal-poset m≥3 integrator fired.
#   'fast_numpy'      = pole-residue closure (m=0/1 native path, or
#                       m≥2 fallback when the analytic path returned
#                       None — typically a degenerate β or mixed
#                       constraint).
#   'fast_callable'   = Sage-compiled fallback (non-local kernel).
from collections import Counter as _Counter
print()
print('=== Phase J evaluator breakdown (per-subset) ===')
phase_j_by_ell = th.get('phase_j_by_ell', {})
grand_ev_counter = _Counter()
grand_m_counter = _Counter()
for ell in sorted(phase_j_by_ell.keys()):
    td_result = phase_j_by_ell[ell]
    groups = td_result.get('groups', []) if td_result else []
    ell_ev_counter = _Counter()
    ell_m_counter = _Counter()
    for diag in groups:
        for ev in (diag.get('subset_evaluators') or []):
            ell_ev_counter[ev] += 1
        for mv in (diag.get('subset_m_values') or []):
            ell_m_counter[mv] += 1
    print(f'  ell={ell}: evaluators={dict(ell_ev_counter)}, '
          f'm_values={dict(ell_m_counter)}')
    grand_ev_counter += ell_ev_counter
    grand_m_counter += ell_m_counter
print(f'  TOTAL:  evaluators={dict(grand_ev_counter)}, '
      f'm_values={dict(grand_m_counter)}')

# Coverage hints for each analytic path.
n_m2 = grand_m_counter.get(2, 0)
n_polygon = grand_ev_counter.get('polygon_modesum', 0)
n_m3plus = sum(c for m, c in grand_m_counter.items() if m >= 3)
n_poset = grand_ev_counter.get('poset_modesum', 0)
if n_m2 == 0:
    print('  (no m=2 subsets — try max_ell >= 1 to exercise '
          'the polygon path)')
elif n_polygon < n_m2:
    print(f'  polygon_modesum fired on {n_polygon}/{n_m2} m=2 subsets '
          f'({n_m2 - n_polygon} fell back, likely non-local kernel)')
if n_m3plus == 0:
    print('  (no m≥3 subsets — try a config with deeper integrals '
          'to exercise the poset path)')
elif n_poset < n_m3plus:
    print(f'  poset_modesum fired on {n_poset}/{n_m3plus} m≥3 subsets '
          f'({n_m3plus - n_poset} fell back to scipy, likely '
          f'degenerate β or mixed constraint)')

## 4.1 Per-diagram C(τ=0) breakdown (diagnostic)

Walks each diagram's contribution callable and evaluates at τ = 0.
Groups by ell, sorts by |C(0)| within each ell, and aggregates by
the maximum m-value present in the diagram's subset list.

Useful for spotting which (set of) diagrams dominate the wrong-sign
1-loop signal — e.g. if all m≥3-bearing diagrams aggregate to a
large negative value but m≤2 diagrams aggregate small/positive, the
m≥3 path is the regression target.


In [ ]:
# Run AFTER Section 4 (relies on ``th`` from compute_cumulants).
import numpy as np

def _eval_diag_at_tau0(cfn, k_val):
    """Evaluate one diagram's contribution callable at τ=0.

    For k=2 with origin_leaf=0 the call signature is f(t1=0, t2=τ);
    at τ=0 → f(0.0, 0.0).  For k=1 → f(0.0).
    """
    if k_val == 2:
        return complex(cfn(0.0, 0.0))
    if k_val == 1:
        return complex(cfn(0.0))
    raise NotImplementedError(f"k={k_val} not handled by this cell")


def _fmt_c(z):
    """Compact complex formatter:  +1.234e-05 + -3.21e-15j"""
    return f'{z.real:+.6e} {z.imag:+.3e}j'


per_diag = []
for ell, td_result in th['phase_j_by_ell'].items():
    if td_result is None:
        continue
    for d_idx, group in enumerate(td_result['groups']):
        cfn = group.get('contribution')
        if cfn is None:
            continue
        val = _eval_diag_at_tau0(cfn, k)
        m_vals = group.get('subset_m_values', []) or []
        evals  = group.get('subset_evaluators', []) or []
        per_diag.append({
            'ell': ell,
            'd_idx': d_idx,
            'value': val,
            'subset_m_values': m_vals,
            'subset_evaluators': evals,
        })

# Sort: by ell first, then by |C(0)| descending
per_diag.sort(key=lambda r: (r['ell'], -abs(r['value'])))

print('\n=== Per-diagram C(τ=0) breakdown (complex) ===\n')
for ell in sorted({r['ell'] for r in per_diag}):
    rows = [r for r in per_diag if r['ell'] == ell]
    total_ell = sum(r['value'] for r in rows)
    print(f'--- ell = {ell}  ({len(rows)} diagrams)  '
          f'Σ C(0) = {_fmt_c(total_ell)} ---')
    print(f'  {"diag":>5}  {"C(0) (Re + Im·j)":<32} {"|C(0)|":>12}  '
          f'{"subset m-vals":<25}  evaluators')
    for r in rows:
        m_str = ','.join(str(m) for m in r['subset_m_values'])
        ev_str = ','.join(sorted(set(r['subset_evaluators'])))
        print(f'  {r["d_idx"]:>5}  {_fmt_c(r["value"]):<32} '
              f'{abs(r["value"]):.4e}  [{m_str:<23}]  {ev_str}')
    print()

# Aggregate by (ell, max-m): partition diagrams into pure-m≤2 and m≥3-bearing
print('=== Aggregate by ell × max-m of subset list ===')
by_max_m = {}
for r in per_diag:
    max_m = max(r['subset_m_values']) if r['subset_m_values'] else 0
    by_max_m.setdefault((r['ell'], max_m), []).append(r['value'])
for (ell, max_m), vals in sorted(by_max_m.items()):
    s = sum(vals)
    print(f'  ell={ell}, max-m={max_m}: {len(vals):>3} diagrams,  '
          f'Σ C(0) = {_fmt_c(s)}')
print()

# Top-5 |C(0)| dominant diagrams across all ells
print('=== Top 5 dominant |C(τ=0)| diagrams ===')
top = sorted(per_diag, key=lambda r: -abs(r['value']))[:5]
for r in top:
    print(f'  ell={r["ell"]} d={r["d_idx"]:>3}: '
          f'C(0)={_fmt_c(r["value"])},  |C(0)|={abs(r["value"]):.4e},  '
          f'evaluators={sorted(set(r["subset_evaluators"]))}')


## 5. Simulation side

Linear-rate + hard-reset simulator with per-pair filters.  All 2
neurons are stacked into a flat array.  Per Euler step:

$$\lambda_i = \max(a_i \cdot v_i, 0)$$
$$n_i \sim \mathrm{Poisson}(\lambda_i\,\mathrm{d}t)$$
$$\tau_{g,ij}\,\dot F_{ij} + F_{ij} = n_j$$
$$\tau_{v,i}\,\dot v_i = -v_i + E_i + \sum_j W_{ij}\,F_{ij}$$
$$v_i \leftarrow 0 \quad\text{if any spike at } i$$

`W[i, j]` carries the signed coupling (all positive here since the
single population is excitatory).

In [ ]:
import secrets as _secrets

# Pull OU/quartic parameters straight from fundamental — no
# build_sim_arrays plumbing needed (the Hawkes helper is shaped
# for Poisson-process simulators that need per-neuron arrays).
mu  = float(fundamental['mu'])
eps = float(fundamental['eps'])
D   = float(fundamental['D'])
tauc = float(fundamental['tauc'])

# Initialise the simulator at the selected MF saddle so the
# transient is ≈ 0 from the start.  For single-well (mu > 0):
# xstar = 0.  For double-well (mu < 0): xstar = ±√(-μ/ε)
# depending on FIXED_POINT_INDEX.
x_init = float(xstar[0])

# Single-population, single-neuron — flat layout is trivially N=1.
N           = 1
pop_offsets = {'pop': (0, 1)}

print(f'mu={mu}, eps={eps}, D={D}')
print(f'x_init={x_init}')
print(f'N={N}, pop_offsets={pop_offsets}')


In [ ]:
# Discretization
n_steps        = int(T_sim / dt_sim)
bin_size_steps = max(int(round(dt_bin / dt_sim)), 1)
dt_bin_eff     = bin_size_steps * dt_sim
n_bins         = n_steps // bin_size_steps
max_lag_bins   = int(tau_max / dt_bin_eff)
tau_sim_grid   = np.arange(-max_lag_bins, max_lag_bins + 1) * dt_bin_eff

# Only one neuron — all external legs map to flat index 0.
pop_indices = [0] * len(external_fields)
# For continuous-field cumulant estimation, treat 'x' as 'dv'
# (smooth, no factorial correction) in compute_kpoint_slice.
field_types = ['dv'] * len(external_fields)

print(f'n_steps     = {n_steps}')
print(f'n_bins      = {n_bins}')
print(f'dt_bin_eff  = {dt_bin_eff}')
print(f'max_lag_bins = {max_lag_bins}')
print(f'pop_indices = {pop_indices}')
print(f'field_types = {field_types}')

# JIT warmup
_ = sim_ou_quartic_colored_numba(
    int(1000), float(dt_sim),
    mu, eps, D, tauc, x_init,
    int(bin_size_steps), int(100), int(0),
)
print('JIT warmup done.')


In [ ]:
BASE_SEED = _secrets.randbits(31)
C_sim_runs   = []
var_runs     = []   # ⟨x²⟩ per run (single scalar)
mean_x_runs  = []   # ⟨x⟩ per run (should be ~0 for symmetric saddle)
voltage_runs = []   # bin-averaged x trajectories

t0 = time.perf_counter()
for run in range(N_RUNS):
    seed = int(BASE_SEED + run)
    x_bins = sim_ou_quartic_colored_numba(
        int(n_steps), float(dt_sim),
        mu, eps, D, tauc, x_init,
        int(bin_size_steps), int(n_bins), seed,
    )
    # x_bins shape (1, n_bins).  Treat as the 'voltage' channel
    # for compute_kpoint_slice with field_types=['dv'].
    voltage_runs.append(x_bins.mean(axis=1))
    var_runs.append(float((x_bins**2).mean()))
    mean_x_runs.append(float(x_bins.mean()))

    if k >= 2:
        tau_run, C_run = compute_kpoint_slice(
            binned_counts=np.zeros_like(x_bins),  # not used for dv
            dt_bin=float(dt_bin_eff),
            pop_indices=[int(p) for p in pop_indices],
            lag_bins=[0, None],
            max_lag_bins=int(max_lag_bins),
            field_types=field_types,
            voltage_bins=x_bins,
        )
        C_sim_runs.append(C_run)
    print(f'  run {run+1}/{N_RUNS}: ⟨x⟩ = {mean_x_runs[-1]:+.4f}, '
          f'⟨x²⟩ = {var_runs[-1]:.4f}')

if k >= 2:
    C_sim_runs = np.array([np.asarray(c).real if hasattr(c, 'real')
                            else np.asarray(c) for c in C_sim_runs])
    C_sim_mean = C_sim_runs.mean(axis=0)
    C_sim_sem  = C_sim_runs.std(axis=0, ddof=1) / np.sqrt(N_RUNS)
else:
    C_sim_runs = C_sim_mean = C_sim_sem = None

var_sim_mean = float(np.mean(var_runs))
var_sim_sem  = float(np.std(var_runs, ddof=1) / np.sqrt(N_RUNS))
mean_x_sim   = float(np.mean(mean_x_runs))
vmean_sim    = np.array(voltage_runs).mean(axis=0)

print(f'\nSimulation side took {time.perf_counter() - t0:.1f}s '
      f'({N_RUNS} runs × T={T_sim:.0g})')
print(f'  Sim ⟨x⟩   = {mean_x_sim:+.6f}')
print(f'  Sim ⟨x²⟩  = {var_sim_mean:.6f}  (SEM {var_sim_sem:.2e})')

# ── Regime-aware reference for ⟨δx²⟩ ─────────────────────────
# In the single-well regime (mu > 0, xstar = 0) the linear-OU
# formula D/μ is the leading prediction.  In the double-well
# regime (mu < 0) the LINEAR analysis around the chosen well
# gives D/μ_eff where μ_eff = -μ + 3·ε·xstar² is the LOCAL
# curvature at xstar (≡ -σ from linear_stability up to sign).
x_s = float(xstar[0])
if mu > 0:
    mu_eff = mu
    pred   = D / mu_eff
    print(f'  Linear-OU prediction ⟨δx²⟩ = D/μ        = {pred:.6f}  '
          f'(no ε correction)')
else:
    mu_eff = -mu + 3.0 * eps * x_s * x_s    # local curvature
    pred   = D / mu_eff
    print(f'  Local-curvature prediction ⟨δx²⟩ = D/μ_eff = {pred:.6f}')
    print(f'    (μ_eff = -μ + 3·ε·x*² = {mu_eff:.4f} at xstar = {x_s:+.4f})')

# ── HOPPING DETECTOR ─────────────────────────────────────────
# At a non-symmetric saddle (xstar ≠ 0), the theory expansion is
# AROUND ONE WELL.  If the Kramers escape time is short relative
# to T_sim, the simulation visits BOTH wells and the empirical
# variance becomes dominated by inter-well hops (~x*²), bearing
# no resemblance to the per-well theory.  Detect by comparing
# |⟨x⟩| to |xstar|: a trapped sim has |⟨x⟩| ≈ |xstar|; a
# hopping sim has |⟨x⟩| ≈ 0 (symmetric averaging).
if abs(x_s) > 1e-6:                           # double-well regime
    barrier = (mu * mu) / (4.0 * eps) if mu < 0 else 0.0
    deviation = abs(mean_x_sim - x_s) / abs(x_s)
    if deviation > 0.2:
        print()
        print(f'  ⚠️  HOPPING DETECTED: ⟨x⟩sim = {mean_x_sim:+.4f} but '
              f'xstar = {x_s:+.4f}.')
        print(f'      The sim is exploring BOTH wells over T_sim={T_sim:.0g}.')
        print(f'      Theory expands around one well; comparison is invalid.')
        print(f'      Kramers barrier = μ²/(4ε) = {barrier:.2f}, '
              f'escape time ≈ exp(barrier/D) = {2.71828 ** (barrier / D):.1f}.')
        print(f'      Trap by EITHER:')
        print(f'        • smaller D (try {D*0.1:.2f}); KEEP |μ| ≲ 1.5 so '
              f'Phase J stays fast — large |μ| explodes the vertex')
        print(f'          coefficients in the m=3 poset modesum.')
        print(f'        • shorter T_sim (try {T_sim*1e-4:.0g} to stay '
              f'inside escape time, but high SEM)')
        print(f'        Avoid: just increasing |μ|.  Barrier grows but '
              f'Phase J runtime grows steeply with |μ| at ell ≥ 1.')
    else:
        print(f'  ✓ sim trapped at xstar = {x_s:+.4f} (deviation {deviation:.1%}).')


if k == 2:
    mid = len(tau_grid_th) // 2
    print()
    print(f'k=2 picked external fields {external_fields}')
    # Per-loop-order corrections at τ=0.
    ell_corr = {
        ell: float(C_by_ell[ell].real[mid])
        for ell in sorted(C_by_ell.keys())
    }
    C_sim_at_0 = float(C_sim_mean[len(C_sim_mean)//2])
    print(f'  Sim C(τ=0)              = {C_sim_at_0:.6f}')
    cum = 0.0
    for ell, corr in ell_corr.items():
        cum += corr
        tag = 'tree' if ell == 0 else f'+ ell={ell}'
        print(f'  Theory {tag:<14} = {corr:+.6f}'
              f'   (cumulative {cum:.6f})')


## 6. Theory vs simulation

**For k = 2**: per-neuron rate bars (sim vs MF) + the C^(2)(τ)
slice with tree-only and tree+loop curves.  For linear φ + reset
the gap between tree-only and tree+1-loop comes entirely from the
spike-reset's (1, 2) vertex.

**For k = 1**: bar chart showing per-neuron sim, tree (= n*),
tree + loop rate for the configured external field.

In [ ]:
# Per-loop-order cumulative theory curves for k=2 cumulant C(τ).
# C_by_ell[ell] is the contribution from exactly that loop order
# (ell=0 is tree, ell=1 first loop, ...).  Cumulative theory at
# order N is sum_{ell=0..N} C_by_ell[ell].
_max_ell_present = max((e for e in C_by_ell.keys()), default=0)
_C_cumulative = {}
_running = np.zeros_like(C_theory_total)
for ell in range(0, _max_ell_present + 1):
    if ell in C_by_ell:
        _running = _running + C_by_ell[ell].real
    _C_cumulative[ell] = _running.copy()

# Colour ramp: tree dark blue → highest-order red.
_COLOURS = ['#3F00FF', '#1F9FCC', '#E67E22', '#E74C3C', '#8E44AD']

def _label_for_order(ell):
    if ell == 0:
        return 'Theory: tree (linear OU)'
    parts = ['tree'] + [f'{e}-loop' for e in range(1, ell + 1)]
    return 'Theory: ' + ' + '.join(parts)

if k == 1:
    print('k=1: ⟨x⟩ = 0 by symmetry of the xstar=0 saddle.')
    print(f'Sim ⟨x⟩ = {mean_x_sim:+.6f} (should be ~0 within MC noise)')
else:
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.plot(tau_sim_grid, C_sim_mean, color='#1f1f1f', linewidth=1.6,
            label=f'Simulation ({N_RUNS} runs)', alpha=0.9)
    ax.fill_between(tau_sim_grid,
                    C_sim_mean - C_sim_sem,
                    C_sim_mean + C_sim_sem,
                    color='#1f1f1f', alpha=0.15, label='Sim SEM')
    for ell in range(0, _max_ell_present + 1):
        style = '--' if ell == 0 else '-'
        ax.plot(tau_grid_th, _C_cumulative[ell],
                color=_COLOURS[ell % len(_COLOURS)],
                linewidth=1.8, linestyle=style,
                label=_label_for_order(ell))
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_xlabel(r'$\tau$')
    ax.set_ylabel(r'$C^{(2)}(\tau) = \langle x(0)\, x(\tau)\rangle$')
    g_eff = eps * D / mu**2
    ax.set_title(f'OU + cubic nonlinearity: '
                 f'μ={mu}, ε={eps}, D={D}, g_eff={g_eff:.3f}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()


## 7. Numerical residual

**k = 2**: tree-only vs tree+loop residual.  For linear φ + reset,
the loop contribution is purely the reset's (1, 2) vertex — small
but not zero.

**k = 1**: rate-shift residual.

In [ ]:
_max_ell_present = max((e for e in C_by_ell.keys()), default=0)

if k == 1:
    print('k=1 cumulant ⟨x⟩ at xstar=0 saddle is identically zero by ')
    print('parity (x → −x symmetry).  Sim noise:')
    print(f'  Sim ⟨x⟩ = {mean_x_sim:+.6e}')
else:
    C_total_on_sim_grid = np.interp(tau_sim_grid, tau_grid_th,
                                    C_theory_total)
    residual            = C_sim_mean - C_total_on_sim_grid
    peak                = max(abs(C_sim_mean.max()),
                              abs(C_sim_mean.min()))
    rms_rel             = float(np.sqrt(np.mean(residual**2)) / peak)
    max_abs_rel         = float(np.max(np.abs(residual)) / peak)
    sem_peak            = float(
        C_sim_sem[np.argmax(np.abs(C_sim_mean))])

    print(f'Sim ⟨x²⟩ at peak     : {peak:+.4e}')
    print(f'Residual RMS / peak (tree + all loops) : {rms_rel:.3%}')
    print(f'Residual max / peak                    : {max_abs_rel:.3%}')
    print(f'Sim SEM at peak                        : {sem_peak:+.3e} '
          f'({sem_peak / peak:.3%} of peak)')

    # Per-order cumulative residual RMS.
    if _max_ell_present >= 1:
        print()
        print('Per-order residual RMS / peak (sim − cumulative theory):')
        running = np.zeros_like(C_theory_total)
        for ell in range(0, _max_ell_present + 1):
            if ell in C_by_ell:
                running = running + C_by_ell[ell].real
            C_on_sim = np.interp(tau_sim_grid, tau_grid_th, running)
            r = C_sim_mean - C_on_sim
            r_rms = float(np.sqrt(np.mean(r**2)) / peak)
            suffix = '' if ell == 0 else ' (cumulative)'
            print(f'  ell={ell}{suffix:<14}: {r_rms:.3%}')


## 8. (Optional) Save outputs

In [ ]:
# Save / export disabled by default; flip ``SAVE = True`` to run.
SAVE = False

if SAVE:
    out_dir = '../pipeline_outputs/ou_quartic_colored'
    os.makedirs(out_dir, exist_ok=True)
    payload = dict(
        mu=mu, eps=eps, D=D,
        k=k, max_ell=max_ell,
        external_fields=external_fields,
        tau_grid_sim=tau_sim_grid,
        tau_grid_th=tau_grid_th,
        C_sim_mean=C_sim_mean,
        C_sim_sem=C_sim_sem,
        C_theory_total=C_theory_total,
        var_sim_mean=var_sim_mean,
        mean_x_sim=mean_x_sim,
    )
    np.savez(os.path.join(out_dir, 'ou_quartic_compare.npz'), **payload)
    print(f'wrote {out_dir}/ou_quartic_compare.npz')


## Summary

Same scalar quartic double-well Ornstein–Uhlenbeck process as the white-noise notebook,
but driven by **colored (exponentially correlated) noise** with correlation time `tauc`
(Lorentzian autocorrelation). The tree-level $k{=}2$ correlator and its loop corrections
are compared against a direct Euler–Maruyama simulation of the colored-noise SDE.

**Knobs:**
* **`fundamental`** — `mu` (mass; `mu < 0` opens the double well), `eps` (the $x^3$ coupling),
  `D` (noise strength), **`tauc`** (noise correlation time; `tauc → 0` recovers the white-noise limit).
* **`k`, `max_ell`** — correlator order and loop order ($k{=}2$, up to 2-loop here).
* **`FIXED_POINT_INDEX`** — which stable well to expand around when `mu < 0`
  (`None` = METADATA default; `0` = lower, `1` = upper).
* **`tau_max`, `tau_step`** — the $\tau$ grid; **`N_RUNS`, `T_sim`, `dt_sim`, `dt_bin`** — simulation knobs.

> **Phase-J cost (μ < 0):** the analytic Phase-J runtime explodes with $|\mu|$ at $\ell\ge 1$;
> prefer $|\mu|\lesssim 1.5$ (with small `D`) or `max_ell=1` for any non-symmetric regime.